In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/KaggleV2-May-2016.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully!
Shape: (110527, 14)


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


In [2]:
invalid_age = df[df['Age'] < 0]

print("Invalid age records:", len(invalid_age))

display(invalid_age)

Invalid age records: 1


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
99832,4.659432e+14,5775010,F,2016-06-06T08:58:13Z,2016-06-06T00:00:00Z,-1,ROMÃO,0,0,0,0,0,0,No


In [3]:
df = df[df['Age'] >= 0].copy()

print("Dataset shape after removing invalid age:")
print(df.shape)

print("Remaining invalid ages:", (df['Age'] < 0).sum())

Dataset shape after removing invalid age:
(110526, 14)
Remaining invalid ages: 0


In [4]:
print("Total duplicate records:", df.duplicated().sum())

Total duplicate records: 0


In [5]:
df = df.drop(columns=['PatientId', 'AppointmentID'])

print("Dataset shape:", df.shape)
print("Remaining columns:")
print(df.columns.tolist())

Dataset shape: (110526, 12)
Remaining columns:
['Gender', 'ScheduledDay', 'AppointmentDay', 'Age', 'Neighbourhood', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received', 'No-show']


In [6]:
df['ScheduledDay'] = pd.to_datetime(df['ScheduledDay'])
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'])

print(df[['ScheduledDay', 'AppointmentDay']].dtypes)

ScheduledDay      datetime64[ns, UTC]
AppointmentDay    datetime64[ns, UTC]
dtype: object


In [7]:
df['WaitingDays'] = (
    df['AppointmentDay'].dt.normalize() -
    df['ScheduledDay'].dt.normalize()
).dt.days

print(df[['ScheduledDay', 'AppointmentDay', 'WaitingDays']].head())

               ScheduledDay            AppointmentDay  WaitingDays
0 2016-04-29 18:38:08+00:00 2016-04-29 00:00:00+00:00            0
1 2016-04-29 16:08:27+00:00 2016-04-29 00:00:00+00:00            0
2 2016-04-29 16:19:04+00:00 2016-04-29 00:00:00+00:00            0
3 2016-04-29 17:29:31+00:00 2016-04-29 00:00:00+00:00            0
4 2016-04-29 16:07:23+00:00 2016-04-29 00:00:00+00:00            0


In [8]:
print("Waiting Days Summary:")
print(df['WaitingDays'].describe())

print("\nNegative Waiting Days:")
print((df['WaitingDays'] < 0).sum())

print("\nZero Waiting Days:")
print((df['WaitingDays'] == 0).sum())

Waiting Days Summary:
count    110526.000000
mean         10.183794
std          15.255034
min          -6.000000
25%           0.000000
50%           4.000000
75%          15.000000
max         179.000000
Name: WaitingDays, dtype: float64

Negative Waiting Days:
5

Zero Waiting Days:
38562


In [9]:
negative_waiting = df[df['WaitingDays'] < 0]

print("Negative Waiting Days Records:")
display(negative_waiting[['ScheduledDay', 'AppointmentDay', 'WaitingDays', 'No-show']])

Negative Waiting Days Records:


,ScheduledDay,AppointmentDay,WaitingDays,No-show
27033,2016-05-10 10:51:53+00:00,2016-05-09 00:00:00+00:00,-1,Yes
55226,2016-05-18 14:50:41+00:00,2016-05-17 00:00:00+00:00,-1,Yes
64175,2016-05-05 13:43:58+00:00,2016-05-04 00:00:00+00:00,-1,Yes
71533,2016-05-11 13:49:20+00:00,2016-05-05 00:00:00+00:00,-6,Yes
72362,2016-05-04 06:50:57+00:00,2016-05-03 00:00:00+00:00,-1,Yes


In [10]:
df = df[df['WaitingDays'] >= 0].copy()

print("Dataset shape after removing negative waiting days:")
print(df.shape)

print("Remaining negative waiting days:", (df['WaitingDays'] < 0).sum())

Dataset shape after removing negative waiting days:
(110521, 13)
Remaining negative waiting days: 0


In [11]:
df['AppointmentDayOfWeek'] = df['AppointmentDay'].dt.dayofweek

print(df[['AppointmentDay', 'AppointmentDayOfWeek']].head())

             AppointmentDay  AppointmentDayOfWeek
0 2016-04-29 00:00:00+00:00                     4
1 2016-04-29 00:00:00+00:00                     4
2 2016-04-29 00:00:00+00:00                     4
3 2016-04-29 00:00:00+00:00                     4
4 2016-04-29 00:00:00+00:00                     4


In [12]:
print("Appointments by Day of Week:")
print(df['AppointmentDayOfWeek'].value_counts().sort_index())

Appointments by Day of Week:
AppointmentDayOfWeek
0    22713
1    25638
2    25866
3    17246
4    19019
5       39
Name: count, dtype: int64


In [13]:
print("No-show Rate by Waiting Days:")

waiting_analysis = df.groupby('WaitingDays')['No-show'].apply(
    lambda x: (x == 'Yes').mean() * 100
)

print(waiting_analysis.head(20))

No-show Rate by Waiting Days:
WaitingDays
0      4.647062
1     21.350470
2     23.821561
3     23.529412
4     23.270321
5     26.609704
6     24.795640
7     26.681614
8     28.730703
9     27.414330
10    31.631919
11    31.610942
12    31.659193
13    31.866825
14    31.342259
15    33.399867
16    30.495222
17    31.616983
18    30.558276
19    34.770115
Name: No-show, dtype: float64


In [14]:
categorical_columns = ['Gender', 'Neighbourhood', 'No-show']

for col in categorical_columns:
    print(f"\n{col}:")
    print("Unique values:", df[col].nunique())
    print(df[col].value_counts().head(10))


Gender:
Unique values: 2
Gender
F    71836
M    38685
Name: count, dtype: int64

Neighbourhood:
Unique values: 81
Neighbourhood
JARDIM CAMBURI       7717
MARIA ORTIZ          5805
RESISTÊNCIA          4430
JARDIM DA PENHA      3877
ITARARÉ              3514
CENTRO               3334
SANTA MARTHA         3131
TABUAZEIRO           3131
JESUS DE NAZARETH    2853
BONFIM               2773
Name: count, dtype: int64

No-show:
Unique values: 2
No-show
No     88207
Yes    22314
Name: count, dtype: int64


In [16]:
numerical_columns = [
    'Age',
    'Scholarship',
    'Hipertension',
    'Diabetes',
    'Alcoholism',
    'Handcap',
    'SMS_received'
]

for col in numerical_columns:
    print(f"\n{col}:")
    print("Unique values:", sorted(df[col].unique()))


Age:
Unique values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(50), np.int64(51), np.int64(52), np.int64(53), np.int64(54), np.int64(55), np.int64(56), np.int64(57), np.int64(58), np.int64(59), np.int64(60), np.int64(61), np.int64(62), np.int64(63), np.int64(64), np.int64(65), np.int64(66), np.int64(67), np.int64(68), np.int64(69), np.int64

In [17]:
categorical_columns = ['Gender', 'Neighbourhood', 'No-show']

for col in categorical_columns:
    print(f"\n{col}:")
    print("Unique values:", df[col].nunique())
    print(df[col].unique())


Gender:
Unique values: 2
['F' 'M']

Neighbourhood:
Unique values: 81
['JARDIM DA PENHA' 'MATA DA PRAIA' 'PONTAL DE CAMBURI' 'REPÚBLICA'
 'GOIABEIRAS' 'ANDORINHAS' 'CONQUISTA' 'NOVA PALESTINA' 'DA PENHA'
 'TABUAZEIRO' 'BENTO FERREIRA' 'SÃO PEDRO' 'SANTA MARTHA' 'SÃO CRISTÓVÃO'
 'MARUÍPE' 'GRANDE VITÓRIA' 'SÃO BENEDITO' 'ILHA DAS CAIEIRAS'
 'SANTO ANDRÉ' 'SOLON BORGES' 'BONFIM' 'JARDIM CAMBURI' 'MARIA ORTIZ'
 'JABOUR' 'ANTÔNIO HONÓRIO' 'RESISTÊNCIA' 'ILHA DE SANTA MARIA'
 'JUCUTUQUARA' 'MONTE BELO' 'MÁRIO CYPRESTE' 'SANTO ANTÔNIO' 'BELA VISTA'
 'PRAIA DO SUÁ' 'SANTA HELENA' 'ITARARÉ' 'INHANGUETÁ' 'UNIVERSITÁRIO'
 'SÃO JOSÉ' 'REDENÇÃO' 'SANTA CLARA' 'CENTRO' 'PARQUE MOSCOSO'
 'DO MOSCOSO' 'SANTOS DUMONT' 'CARATOÍRA' 'ARIOVALDO FAVALESSA'
 'ILHA DO FRADE' 'GURIGICA' 'JOANA D´ARC' 'CONSOLAÇÃO' 'PRAIA DO CANTO'
 'BOA VISTA' 'MORADA DE CAMBURI' 'SANTA LUÍZA' 'SANTA LÚCIA'
 'BARRO VERMELHO' 'ESTRELINHA' 'FORTE SÃO JOÃO' 'FONTE GRANDE'
 'ENSEADA DO SUÁ' 'SANTOS REIS' 'PIEDADE' 'JESUS DE NAZARE

In [18]:
df = df.drop(columns=['ScheduledDay', 'AppointmentDay'])

print("Final shape:", df.shape)
print("Remaining columns:")
print(df.columns.tolist())

Final shape: (110521, 12)
Remaining columns:
['Gender', 'Age', 'Neighbourhood', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received', 'No-show', 'WaitingDays', 'AppointmentDayOfWeek']


In [19]:
print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nNegative ages:", (df['Age'] < 0).sum())
print("Negative waiting days:", (df['WaitingDays'] < 0).sum())

Dataset shape: (110521, 12)

Missing values:
Gender                  0
Age                     0
Neighbourhood           0
Scholarship             0
Hipertension            0
Diabetes                0
Alcoholism              0
Handcap                 0
SMS_received            0
No-show                 0
WaitingDays             0
AppointmentDayOfWeek    0
dtype: int64

Duplicate rows: 18414

Negative ages: 0
Negative waiting days: 0


In [20]:
df.to_csv("../data/cleaned_hospital_appointments.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!
